In [1]:
# %%
import cv2
import numpy as np
import tensorflow as tf
import gc
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback

# ✅ Limit VRAM Usage to 6GB
gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.experimental.set_virtual_device_configuration(
            gpus[0],
            [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=16384)]  # 16GB limit
        )
        print("VRAM limited to 6GB")
    except RuntimeError as e:
        print(e)

# Enable Mixed Precision (Reduces VRAM usage)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# %%
train_dir = "train"
valid_dir = "valid"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Reduce batch size to save memory

# %%
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

class_labels = list(train_generator.class_indices.keys())
print(class_labels)

# %%
class FreeMemoryCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        tf.keras.backend.clear_session()
        gc.collect()
        print(f"Cleared memory after epoch {epoch+1}.")

# %%
base_model = ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(len(train_generator.class_indices), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=x)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# %%
checkpoint = ModelCheckpoint(
    "best_model.h5", monitor="val_accuracy", save_best_only=True, mode="max", verbose=1
)

early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
free_memory = FreeMemoryCallback()

# %%
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=30,
    callbacks=[early_stopping, checkpoint, free_memory]
)

# %%
loss, acc = model.evaluate(valid_generator)
print(f"ResNet Model Accuracy: {acc * 100:.2f}%")

# %%
def save_as_tflite(model_path, model_name):
    model = tf.keras.models.loa

2025-02-14 13:53:21.365268: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-14 13:53:21.372599: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739521401.380623  101159 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739521401.382955  101159 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-14 13:53:21.391479: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

VRAM limited to 6GB
Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Lea

I0000 00:00:1739521404.806143  101159 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 16384 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
/home/vani/Desktop/Projects/CropDiseaseDetection1345/.venv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


I0000 00:00:1739521416.785804  101359 service.cc:148] XLA service 0x76ce7c0141c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1739521416.785819  101359 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-02-14 13:53:37.022422: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1739521418.984562  101359 cuda_dnn.cc:529] Loaded cuDNN version 90600


...

2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.0639 - loss: 3.6184

/home/vani/Desktop/Projects/CropDiseaseDetection1345/.venv/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2025-02-14 13:58:20.360366: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1928', 8 bytes spill stores, 8 bytes spill loads

2025-02-14 13:58:20.490815: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1935', 8 bytes spill stores, 8 bytes spill loads




Epoch 1: val_accuracy improved from -inf to 0.02715, saving model to best_model.h5


Cleared memory after epoch 1.
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 296s 122ms/step - accuracy: 0.0639 - loss: 3.6182 - val_accuracy: 0.0271 - val_loss: 10.9242
Epoch 2/30
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.2259 - loss: 2.5708
Epoch 2: val_accuracy did not improve from 0.02715
Cleared memory after epoch 2.
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 251s 114ms/step - accuracy: 0.2259 - loss: 2.5707 - val_accuracy: 0.0207 - val_loss: 13.1848
Epoch 3/30
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.3053 - loss: 2.2200
Epoch 3: val_accuracy improved from 0.02715 to 0.13527, saving model to best_model.h5


Cleared memory after epoch 3.
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 250s 113ms/step - accuracy: 0.3053 - loss: 2.2199 - val_accuracy: 0.1353 - val_loss: 6.5133
Epoch 4/30
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.3936 - loss: 1.9068
Epoch 4: val_accuracy did not improve from 0.13527
Cleared memory after epoch 4.
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 248s 113ms/step - accuracy: 0.3936 - loss: 1.9068 - val_accuracy: 0.0273 - val_loss: nan
Epoch 5/30
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.4217 - loss: 1.8206
Epoch 5: val_accuracy did not improve from 0.13527
Cleared memory after epoch 5.
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 251s 114ms/step - accuracy: 0.4218 - loss: 1.8206 - val_accuracy: 0.0242 - val_loss: nan
Epoch 6/30
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.4518 - loss: 1.7334
Epoch 6: val_accuracy did not improve from 0.13527
Cleared memory after epoch 6.
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 248s 113ms/step - accuracy: 0.4518 - loss: 1.7334 - val_accurac